# 4-class balanced training, YOLO11s: Live_Knot, Dead_Knot, Resin, Crack
Top 4 classes by box count in dataset_clean. Three training sets built here by build_balanced.py (seed 42), same val/test:
- **plain**: real train images, 4-class labels (control)
- **full**: + grain-aware copy-paste until every class has n_max boxes
- **sqrt**: + copy-paste until class c has sqrt(n_c * n_max) boxes (repeat-factor rule)

Config identical to the YOLO11s baseline (run 3): yolo11s.pt, 1024 rect, batch 32, lr0 0.001, patience 40, seed 42.

**Pre-registered rule (fixed before any result):** between full and sqrt, the one with the higher 4-class *val* mAP50 is the balanced model.
The test split is then evaluated exactly once for the plain control and the chosen balanced model. No decision uses test numbers.

In [ ]:
!pip -q install ultralytics==8.4.142
!nvidia-smi --query-gpu=name,memory.total --format=csv
import ultralytics, torch, os, glob, shutil, time, json, subprocess, sys
T0 = time.time()
print(ultralytics.__version__, torch.__version__, torch.cuda.device_count(), 'GPUs')
yml = glob.glob('/kaggle/input/**/dataset_clean/data.yaml', recursive=True)
assert yml, os.listdir('/kaggle/input')
SRC = os.path.dirname(yml[0])
hits = glob.glob('/kaggle/input/**/build_balanced.py', recursive=True)
assert hits, 'build_balanced.py not found - is wood-defects-scripts (v4+) attached?'
SCRIPTS = os.path.dirname(hits[0]); print('scripts dir:', SCRIPTS, os.listdir(SCRIPTS))
DATA = '/kaggle/tmp/dataset_clean'
if not os.path.isdir(DATA + '/train/images'):
    t = time.time(); shutil.copytree(SRC, DATA); print(f'copied dataset to local disk in {time.time()-t:.0f}s')

In [ ]:
# build the 4-class plain / full / sqrt training sets (train split only; val and test are real images)
OUT = '/kaggle/tmp/dataset_4cls'
t = time.time()
rc = subprocess.call([sys.executable, f'{SCRIPTS}/build_balanced.py', '--src', DATA, '--out', OUT, '--seed', '42'])
assert rc == 0, 'build_balanced.py failed'
shutil.copytree(f'{OUT}/previews', '/kaggle/working/previews', dirs_exist_ok=True)
shutil.copy2(f'{OUT}/balance_report.json', '/kaggle/working/balance_report.json')
for v in ['plain', 'full', 'sqrt']:
    print(open(f'{OUT}/data_{v}.yaml').read())
print(f'built in {time.time()-t:.0f}s')

In [ ]:
COMMON = ['--device', '0', '--model', 'yolo11s.pt', '--batch', '32', '--lr0', '0.001', '--patience', '40',
          '--project', '/kaggle/working/outputs', '--persist', '/kaggle/working/persist']
RUNS = [('full', 'yolo11s_4cls_full'), ('plain', 'yolo11s_4cls_plain'), ('sqrt', 'yolo11s_4cls_sqrt')]
done, hours = {}, {}
for v, name in RUNS:
    el = (time.time() - T0) / 3600
    if el > 8.0:
        print('SKIP', name, f'elapsed {el:.2f} h'); continue
    cmd = [sys.executable, f'{SCRIPTS}/train_yolo11m.py', '--data', f'{OUT}/data_{v}.yaml', '--name', name] + COMMON
    print(' '.join(cmd)); t = time.time(); rc = subprocess.call(cmd)
    hours[name] = round((time.time() - t) / 3600, 2)
    ok = os.path.exists(f'/kaggle/working/outputs/{name}/weights/best.pt')
    print(name, 'exit', rc, 'best.pt OK' if ok else 'NO best.pt', hours[name], 'h')
    if ok:
        done[v] = name
        print(open(f'/kaggle/working/outputs/{name}/results.csv').read()[-600:])
json.dump(dict(done=done, hours=hours), open('/kaggle/working/runs_done.json', 'w'), indent=2)
print(done, hours)

In [ ]:
# validation of every model, then the pre-registered choice between full and sqrt
val = {}
for v, name in done.items():
    out = f'/kaggle/working/eval/val_{name}'
    subprocess.call([sys.executable, f'{SCRIPTS}/evaluate.py', '--weights', f'/kaggle/working/outputs/{name}/weights/best.pt',
                     '--data', f'{OUT}/data_{v}.yaml', '--split', 'val', '--device', '0', '--out', out])
    md = open(f'{out}/val_metrics.md').read(); print(name); print(md)
    val[v] = float([l.split('|') for l in md.splitlines() if l.startswith('| ALL')][0][5])
cands = {v: m for v, m in val.items() if v in ('full', 'sqrt')}
CHOSEN = max(cands, key=cands.get) if cands else None
json.dump(dict(val_map50=val, chosen=CHOSEN, rule='higher 4-class val mAP50 of full vs sqrt'),
          open('/kaggle/working/selection.json', 'w'), indent=2)
print('val mAP50:', val, '-> chosen balanced model:', CHOSEN)

## Test evaluation: exactly once per model (plain control + chosen balanced). No decision is made from these numbers.

In [ ]:
for v in ['plain', CHOSEN]:
    if v is None or v not in done:
        continue
    name = done[v]; out = f'/kaggle/working/eval/test_{name}'
    subprocess.call([sys.executable, f'{SCRIPTS}/evaluate.py', '--weights', f'/kaggle/working/outputs/{name}/weights/best.pt',
                     '--data', f'{OUT}/data_{v}.yaml', '--split', 'test', '--device', '0', '--out', out])
    print(name); print(open(f'{out}/test_metrics.md').read())
print(f'total wall-clock {(time.time() - T0) / 3600:.2f} h')